# SBBTS – Step-by-step walkthrough on SPX

Each section exposes **one building block** and prints what it does.
All graph utilities from `sbbts.utils.visualization` are exercised in §22 onward.

Edit the three cells below before running anything else.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════
N_DAYS = 1260   # tail of price history to keep  (1260 ≈ 5 trading years)
T      = 252    # window length in time steps     (252 = 1 trading year)
MODE   = 'LITE' # 'LITE'  →  quick CPU run (~10-20 min)
                # 'FULL'  →  paper-quality  (~8 h CPU / ~15 min A100)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"N_DAYS = {N_DAYS}  ({N_DAYS/252:.1f} trading years)")
print(f"T      = {T}  ({T/252:.1f} trading year per window)")
print(f"MODE   = {MODE!r}")
print(f"Windows available: {N_DAYS - T} (≈ {(N_DAYS - T)/T:.1f} years of non-overlapping data)")


## 0 · Imports


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "..")

# from finance.data.prices import get_prices  # private — uncomment with the cell below

# Transport-map primitives
from sbbts.transport.transport_map import x_to_y, y_to_x, validate_beta_condition

# Brownian bridge
from sbbts.transport.brownian_bridge import (
    brownian_bridge_mean,
    brownian_bridge_std,
    sample_brownian_bridge,
    sample_brownian_bridge_batch,
)

# Conditional OT (inner training-loop logic)
from sbbts.transport.conditional_ot import (
    compute_conditional_transport,
    sample_training_points,
    compute_score_target,
    compute_interval_loss,
)

# Risk metrics
from sbbts.utils.metrics import (
    compute_returns,
    var,
    expected_shortfall,
    sharpe_ratio,
    autocorrelation,
    compute_all_risk_metrics,
    log_ret_to_prices,
    compute_metrics,
    compute_tstr,
)

# SDE simulation
from sbbts.utils.sampling import (
    euler_maruyama_step,
    generate_brownian_motion,
    generate_gbm,
)

# All visualization utilities
from sbbts.utils.visualization import (
    plot_sample_paths,
    plot_marginal_comparison,
    plot_acf_comparison,
    plot_acf_vol,
    plot_qq,
    plot_risk_metrics,
    plot_lag_corr_matrix,
    plot_leverage_effect,
    plot_rolling_vol,
    plot_cluster_diagnostics,
    plot_signature_moments,
    plot_correlation_comparison,
    tstr_score,
    diagnose,
    full_diagnose,
)

# Logger
from sbbts.utils.logger import SBBTSLogger

# Top-level model class
from sbbts.core.sbbts_solver import SBBTS

print("imports OK")


imports OK


---
## 1 · Load & crop SPX

`tail(N_DAYS)` gives us the last `N_DAYS` close prices (from global params).


In [ ]:
# #── Private data source ─────────────────────────────────────────────────────
# #Uncomment this cell and comment the next one if you have internal data access.

# from finance.data.prices import get_prices

# spx_prices = get_prices(["^GSPC"], period="20y")
# spx = spx_prices['^GSPC']['close'].tail(N_DAYS)

# print(f"Date range   : {spx.index[0].date()} → {spx.index[-1].date()}")
# #─────────────────────────────────────────────────────────────────────────────

Date range   : 2021-06-08 → 2026-06-12


In [ ]:
# ── Public data source (yfinance) ────────────────────────────────────────────
try:
    import yfinance as yf
    spx_raw = yf.download("^GSPC", period="10y", auto_adjust=True, progress=False)["Close"]
    spx = spx_raw.dropna().squeeze().tail(N_DAYS)
    print(f"SPX loaded via yfinance: {len(spx)} days")
except Exception as e:
    print(f"yfinance unavailable ({e}) — falling back to GBM simulation")
    np.random.seed(42)
    log_rets_sim = np.random.normal(0.0003, 0.01, N_DAYS)
    spx = pd.Series(4000 * np.exp(np.cumsum(log_rets_sim)), name="Close")
    print(f"GBM fallback: {len(spx)} days")

In [ ]:
print(f"N prices     : {len(spx)}")
print(f"T (window)   : {T}   ->  N_windows = {len(spx) - T}")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(spx.values, color="#1565C0", lw=0.8)
ax.set_title(f"SPX - last {N_DAYS} close prices")
ax.set_ylabel("Price")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# ─────────────────────────────────────────────────────────────────────────────

---
## 2 · `compute_returns` — log returns from prices

```
sbbts/utils/metrics.py :: compute_returns(prices, log_returns=True)
```

Computes `r_t = log(P_t / P_{t-1})` along the last axis.
`N_DAYS` prices → **N_DAYS-1 log returns**.


In [ ]:
prices_np = spx.values          # shape (N_DAYS,)

log_rets = compute_returns(prices_np, log_returns=True)
print(f"Input  shape : {prices_np.shape}")
print(f"Output shape : {log_rets.shape}   (one less — can't compute return at t=0)")
print(f"Mean return  : {log_rets.mean():.6f}   ({log_rets.mean()*252:.2%} annualised)")
print(f"Std  return  : {log_rets.std():.6f}   ({log_rets.std()*np.sqrt(252):.2%} ann. vol)")

manual = np.diff(np.log(prices_np))
assert np.allclose(log_rets, manual)
print("Matches np.diff(np.log(prices)): ✓")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(spx.index[1:], log_rets, lw=0.5, color='#C62828', alpha=0.8)
ax.set_title(f"SPX log returns — last {N_DAYS-1} days")
ax.set_ylabel("Log return")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3 · Roll into windows → shape `(N, T, d)`

SBBTS models **trajectories** — windows of `T` consecutive returns.
Sliding window with stride 1, using `T` from global params.

- `N` = number of windows
- `T` = time-steps per window  (252 = 1 trading year)
- `d` = number of assets (1 here — univariate SPX)


In [ ]:
d = 1       # dimension (univariate SPX)

windows = np.lib.stride_tricks.sliding_window_view(log_rets, T)  # (N, T)
X_train = windows[:, :, np.newaxis].astype(np.float32)           # (N, T, 1)

N = X_train.shape[0]

print(f"log_rets length : {len(log_rets)}")
print(f"T (from params) : {T}")
print(f"X_train shape   : {X_train.shape}   →  (N={N}, T={T}, d={d})")
print(f"Non-overlapping windows ≈ {N // T:.0f}  ({N} overlapping)")

assert np.allclose(X_train[0, :, 0], log_rets[:T])
print("First window matches log_rets[0:T]: ✓")

# Plot a few windows
fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
for i, ax in enumerate(axes):
    ax.plot(X_train[i*N//3, :, 0], lw=0.8, color='#1565C0')
    ax.set_title(f"Window {i*N//3}")
    ax.set_xlabel("day in window")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("log return")
plt.suptitle("Three sample windows from X_train", y=1.02)
plt.tight_layout()
plt.show()


---
## 4 · `SBBTS.suggest_beta` — automatic β selection

```
sbbts/core/sbbts_solver.py :: SBBTS.suggest_beta(n_time_points, T, safety_factor)
```

**Theorem 3.2** requires `β · Δt > 1` on every interval.
With `T` time-points and terminal time `T_end=1.0`:

```
Δt = T_end / (T - 1)
β  = safety_factor / Δt   =   safety_factor × (T - 1)
```

With T=252 and safety_factor=5: β ≈ 1255. Large β = strong drift regularisation.


In [ ]:
T_end = 1.0     # always normalise time to [0, 1]
dt = T_end / (T - 1)

beta_suggested = SBBTS.suggest_beta(n_time_points=T, T=T_end, safety_factor=5.0)
beta_minimum   = SBBTS.suggest_beta(n_time_points=T, T=T_end, safety_factor=1.0)

print(f"T time-points  : {T}")
print(f"Δt             : {dt:.6f}  (= {T_end} / {T-1})")
print(f"β minimum      : {beta_minimum:.1f}   (β·Δt = 1.00 — exact boundary)")
print(f"β suggested    : {beta_suggested:.1f}  (β·Δt = 5.00 — comfortable margin)")
print(f"\nWe'll use β = {beta_suggested:.0f} throughout this notebook.")

BETA = float(beta_suggested)


---
## 5 · `validate_beta_condition` — per-interval check

```
sbbts/transport/transport_map.py :: validate_beta_condition(beta, dt, interval_idx)
```

Checks `β * Δt > 1` and raises `ValueError` if not.
Called once per interval in the solver; here we call it manually.


In [ ]:
validate_beta_condition(beta=BETA, dt=dt, interval_idx=0)
print(f"β={BETA:.0f}, Δt={dt:.6f}  →  β·Δt = {BETA*dt:.2f}  ✓  (> 1)")

try:
    validate_beta_condition(beta=5.0, dt=dt, interval_idx=0)
except ValueError as e:
    print(f"\nExpected error with β=5: {e}")


---
## 6 · Time grid — `torch.linspace`

Internally this is just `torch.linspace(0, T_end, T)`.
Creates the T evenly-spaced time-stamps `[t_0=0, t_1, ..., t_{T-1}=1]`
at which we "observe" the path.


In [ ]:
time_points = torch.linspace(0, T_end, T)

print(f"time_points shape  : {time_points.shape}")
print(f"First 5            : {time_points[:5].tolist()}")
print(f"Last  5            : {time_points[-5:].tolist()}")
print(f"Δt (uniform)       : {(time_points[1] - time_points[0]).item():.6f}")


---
## 7 · `x_to_y` & `y_to_x` — the transport map

```
sbbts/transport/transport_map.py :: x_to_y(x, score, beta)
                                    y_to_x(y, score, beta)
```

The SBBTS change of measure:
```
Y = X  −  (1/β) · s_θ(t, X, context)    ← x_to_y
X = Y  +  (1/β) · s_θ(t, Y, context)    ← y_to_x
```
Without a trained score network → zero correction → Y = X (identity).


In [ ]:
batch_size = 8
x = torch.tensor(X_train[:batch_size, 0, :])   # shape (8, 1)

# Zero score (untrained network)
score_zero = torch.zeros_like(x)
y = x_to_y(x, score_zero, beta=BETA)
print("x_to_y with zero score  →  Y = X − 0 = X")
print(f"  max |y − x| : {(y - x).abs().max().item():.2e}  (= 0 as expected)\n")

# Small non-zero score
score_small = torch.full_like(x, 0.01)
y_shifted   = x_to_y(x, score_small, beta=BETA)
x_recovered = y_to_x(y_shifted, score_small, beta=BETA)

print(f"x_to_y then y_to_x  →  x_recovered == x ? {torch.allclose(x, x_recovered)} ✓")
print(f"  shift per step : {(1/BETA * score_small[0,0]).item():.8f}  (= 1/β · score)")
print(f"  (small because β={BETA:.0f} is large)")


---
## 8 · `brownian_bridge_mean` & `brownian_bridge_std`

```
sbbts/transport/brownian_bridge.py :: brownian_bridge_mean(y_start, y_end, t, t_i, t_i1)
                                      brownian_bridge_std(t, t_i, t_i1)
```

A Brownian bridge is pinned at both endpoints.
Given `y_start = Y_{t_i}` and `y_end = Y_{t_{i+1}}`, at inner time `t`:

```
mean  =  ((t_{i+1} - t) / Δt) · y_start  +  ((t - t_i) / Δt) · y_end
std²  =  (t - t_i)(t_{i+1} - t) / Δt
```


In [ ]:
t_i  = time_points[0].item()   # 0.0
t_i1 = time_points[1].item()   # 1/(T-1) ≈ 0.004 with T=252

y_start = torch.tensor(X_train[:batch_size, 0, :])
y_end   = torch.tensor(X_train[:batch_size, 1, :])

# Mean at midpoint
t_mid    = torch.full((batch_size,), (t_i + t_i1) / 2)
mean_mid = brownian_bridge_mean(y_start, y_end, t_mid, t_i, t_i1)
print(f"Bridge mean at midpoint == (y_start + y_end)/2:")
print(f"  manual   : {((y_start + y_end)/2).squeeze().tolist()[:4]}...")
print(f"  function : {mean_mid.squeeze().tolist()[:4]}...")
print(f"  max diff : {((mean_mid - (y_start+y_end)/2)).abs().max().item():.2e}  ✓\n")

# Std profile over one interval
t_scan   = torch.linspace(t_i, t_i1, 100)
std_scan = [brownian_bridge_std(torch.tensor(tv), t_i, t_i1).item() for tv in t_scan]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_scan.numpy(), std_scan, color='#1565C0')
ax.set_xlabel("t")
ax.set_ylabel("std(Y_t)")
ax.set_title(f"Bridge std over first interval [{t_i:.4f}, {t_i1:.4f}]  (Δt={dt:.4f})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 9 · `sample_brownian_bridge` — draw one bridge sample

```
sbbts/transport/brownian_bridge.py :: sample_brownian_bridge(y_start, y_end, t, t_i, t_i1)
```

Draws `Y_t ~ W | Y_{t_i}=y_start, Y_{t_{i+1}}=y_end`:

```
Y_t = mean(t) + std(t) · Z,    Z ~ N(0, I)
```

This is the training point sampled by Algorithm 1 — no full SDE needed.


In [ ]:
torch.manual_seed(42)

y_t_sample = sample_brownian_bridge(y_start, y_end, t_mid, t_i, t_i1)
print(f"y_start  (day 0) : {y_start.squeeze().tolist()[:4]}...")
print(f"y_t (mid interval): {y_t_sample.squeeze().tolist()[:4]}...")
print(f"y_end    (day 1) : {y_end.squeeze().tolist()[:4]}...")

# Visualise 20 bridge realisations for the first window
t_fine = torch.linspace(t_i, t_i1, 80)
y0 = y_start[0]
y1 = y_end[0]

fig, ax = plt.subplots(figsize=(10, 4))
for _ in range(20):
    pts = [sample_brownian_bridge(y0.unsqueeze(0), y1.unsqueeze(0),
                                  torch.tensor([tv.item()]), t_i, t_i1).item()
           for tv in t_fine]
    ax.plot(t_fine.numpy(), pts, alpha=0.35, lw=0.8, color='steelblue')
ax.scatter([t_i, t_i1], [y0.item(), y1.item()], color='red', zorder=5, s=60, label='endpoints')
ax.set_title("20 Brownian bridge realisations over interval [t_0, t_1]")
ax.set_xlabel("t")
ax.set_ylabel("Y_t")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 10 · `sample_brownian_bridge_batch` — random time + bridge value

```
sbbts/transport/brownian_bridge.py :: sample_brownian_bridge_batch(y_start, y_end, t_i, t_i1)
```

Algorithm 1 training loop sampling:
1. Draw `t ~ Uniform[t_i, t_{i+1}]` per batch element
2. Draw `Y_t` from the bridge at that time

Returns `(t, Y_t)` — both random.


In [ ]:
torch.manual_seed(0)

t_sampled, y_t_sampled = sample_brownian_bridge_batch(
    y_start, y_end, t_i=t_i, t_i1=t_i1, n_samples=1
)

print(f"t_sampled   shape : {t_sampled.shape}   (one random time per batch element)")
print(f"y_t_sampled shape : {y_t_sampled.shape} (bridge value at that time)")
print(f"\nt_sampled  : {t_sampled.tolist()}")
print(f"y_t_sampled: {y_t_sampled.squeeze().tolist()}")
print(f"\nt in [{t_i:.6f}, {t_i1:.6f}]: {all(t_i <= tv <= t_i1 for tv in t_sampled.tolist())} ✓")


---
## 11 · `compute_conditional_transport` — map X to Y at both endpoints

```
sbbts/transport/conditional_ot.py :: compute_conditional_transport(...)
```

Algorithm 1, lines 5-6.
Given `X_{t_i}` and `X_{t_{i+1}}` (real data), computes `Y_{t_i}` and `Y_{t_{i+1}}`:
```
Y_ti  = X_ti  − (1/β) · s_θ(t_i,   X_ti,  context)
Y_ti1 = X_ti1 − (1/β) · s_θ(t_i1, X_ti1, context)
```


In [ ]:
x_ti  = torch.tensor(X_train[:batch_size, 0, :])   # (8, 1)
x_ti1 = torch.tensor(X_train[:batch_size, 1, :])   # (8, 1)

def dummy_score_fn(t, x, context=None):
    return torch.zeros_like(x)

y_ti_out, y_ti1_out, score_ti_out, score_ti1_out = compute_conditional_transport(
    x_ti, x_ti1,
    score_fn=dummy_score_fn,
    beta=BETA,
    t_i=t_i,
    t_i1=t_i1,
    context=None,
)

print("x_to_y with zero score → Y = X:")
print(f"  x_ti   : {x_ti.squeeze().tolist()[:4]}...")
print(f"  y_ti   : {y_ti_out.squeeze().tolist()[:4]}...")
print(f"  max |y_ti - x_ti|   : {(y_ti_out - x_ti).abs().max().item():.2e}")
print(f"  max |y_ti1 - x_ti1| : {(y_ti1_out - x_ti1).abs().max().item():.2e}")


---
## 12 · `sample_training_points` + `compute_score_target`

```
sbbts/transport/conditional_ot.py :: sample_training_points(y_ti, y_ti1, t_i, t_i1)
                                     compute_score_target(y_ti1, y_t, t, t_i1)
```

Algorithm 1, lines 7-8.
`compute_score_target` computes what the network should predict:
```
target = (Y_{t_{i+1}} − Y_t) / (t_{i+1} − t)
```
This is the direction from the current bridge position toward the endpoint.


In [ ]:
torch.manual_seed(7)

t_train, y_t_train = sample_training_points(y_ti_out, y_ti1_out, t_i=t_i, t_i1=t_i1)
target = compute_score_target(y_ti1_out, y_t_train, t_train, t_i1=t_i1)

print(f"t_train shape   : {t_train.shape}")
print(f"y_t_train shape : {y_t_train.shape}")
print(f"target shape    : {target.shape}")
print(f"\ntarget values   : {target.squeeze().tolist()[:4]}...")
print(f"  = (y_ti1 - y_t) / (t_i1 - t)  →  points toward endpoint")

# Manual check [0]
manual_target_0 = (y_ti1_out[0] - y_t_train[0]) / (t_i1 - t_train[0].item())
print(f"\nManual check [0]: {manual_target_0.item():.6f}  vs  {target[0,0].item():.6f}  ✓")


---
## 13 · `compute_interval_loss` — score matching MSE

```
sbbts/transport/conditional_ot.py :: compute_interval_loss(score_pred, score_target)
```

Equation (4.1):
```
L_i = E [ ||s_θ(t, Y_t, context) − target||² ]
```
At initialisation (s_θ ≡ 0): loss = ||target||².


In [ ]:
score_pred = dummy_score_fn(t_train, y_t_train)
loss = compute_interval_loss(score_pred, target)

print(f"score_pred (untrained, zeros) : {score_pred.squeeze().tolist()[:4]}...")
print(f"target                        : {target.squeeze().tolist()[:4]}...")
print(f"\nInterval loss = ||pred − target||²  = {loss.item():.6f}")

manual_loss = ((score_pred - target)**2).sum(dim=-1).mean().item()
print(f"Manual check                       = {manual_loss:.6f}  ✓")


---
## 14 · `var` & `expected_shortfall` — tail risk

```
sbbts/utils/metrics.py :: var(returns, confidence)
                          expected_shortfall(returns, confidence)
```

**VaR_α** = the α-quantile of the **loss** distribution
**ES_α** = mean loss given loss exceeds VaR_α  (heavier tail metric)


In [ ]:
var_95 = var(log_rets, confidence=0.95)
var_99 = var(log_rets, confidence=0.99)
es_95  = expected_shortfall(log_rets, confidence=0.95)
es_99  = expected_shortfall(log_rets, confidence=0.99)

print(f"VaR  95% : {var_95:.4f}  ({var_95*100:.2f}% daily loss, not exceeded 95% of days)")
print(f"VaR  99% : {var_99:.4f}")
print(f"ES   95% : {es_95:.4f}  (avg loss on worst 5% of days)")
print(f"ES   99% : {es_99:.4f}  (avg loss on worst 1% of days)")
print(f"\nES > VaR because ES averages the full tail beyond VaR.")

# Manual VaR 95%
manual_var95 = -np.percentile(log_rets, 5)
print(f"\nManual VaR 95% = -np.percentile(rets, 5) = {manual_var95:.4f}  ✓")


---
## 15 · `sharpe_ratio` — risk-adjusted return

```
sbbts/utils/metrics.py :: sharpe_ratio(returns, annualization_factor=252)
```

```
Sharpe = (mean_daily_return / std_daily_return) × sqrt(252)
```


In [ ]:
sr = sharpe_ratio(log_rets, annualization_factor=252, risk_free_rate=0.0)

print(f"Annualised Sharpe ratio : {sr:.4f}")
print(f"  = mean({log_rets.mean():.6f}) / std({log_rets.std(ddof=1):.6f}) × sqrt(252)")
print(f"  Rule of thumb: >1 decent, >2 good")

manual_sr = (log_rets.mean() / log_rets.std(ddof=1)) * np.sqrt(252)
print(f"Manual check            : {manual_sr:.4f}  ✓")


---
## 16 · `autocorrelation` — serial dependence

```
sbbts/utils/metrics.py :: autocorrelation(series, max_lag=20)
```

Returns are nearly uncorrelated (EMH), but **|returns|** show strong positive ACF
(volatility clustering). SBBTS should replicate the second pattern.


In [ ]:
acf_returns  = autocorrelation(log_rets,        max_lag=20)
acf_abs_rets = autocorrelation(np.abs(log_rets), max_lag=20)

lags = np.arange(21)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(lags, acf_returns,  color='#1565C0', alpha=0.8)
axes[0].axhline(0, color='k', lw=0.8)
axes[0].set_title("ACF of log returns  (should be near zero — EMH)")
axes[0].set_xlabel("Lag")
axes[0].grid(True, alpha=0.3)

axes[1].bar(lags, acf_abs_rets, color='#C62828', alpha=0.8)
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_title("ACF of |log returns|  (positive = volatility clustering)")
axes[1].set_xlabel("Lag")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"ACF[1] returns   = {acf_returns[1]:.4f}   (near zero = no autocorrelation in returns)")
print(f"ACF[1] |returns| = {acf_abs_rets[1]:.4f}   (positive = vol clusters)")


---
## 17 · `compute_all_risk_metrics` — one-call summary

```
sbbts/utils/metrics.py :: compute_all_risk_metrics(returns, annualization_factor=252)
```

Bundles VaR, ES, annualised return & vol, and Sharpe into a single dict.
Used by `diagnose()` to compare real vs synthetic (Table 4 in the paper).


In [ ]:
metrics = compute_all_risk_metrics(log_rets, annualization_factor=252)

print(f"{'Metric':<18} {'Value':>12}")
print("-" * 32)
for k, v in metrics.items():
    print(f"  {k:<16} {v:>12.4f}")


---
## 18 · `euler_maruyama_step` — one SDE step

```
sbbts/utils/sampling.py :: euler_maruyama_step(x, t, dt, drift_fn, diffusion_fn)
```

Discretises `dX = μ(X,t) dt + σ(X,t) dW`:
```
X_{t+dt} = X_t + μ(X_t, t)·dt + σ(X_t, t)·√dt·Z,   Z~N(0,I)
```

In SBBTS's `sample()` this runs `n_euler_steps` times per interval.


In [ ]:
torch.manual_seed(1)

x0 = torch.tensor(X_train[:batch_size, 0, :])   # (8, 1)
dt_step = dt / 50                                 # 50 sub-steps per interval

drift_zero = lambda x, t: torch.zeros_like(x)

x1 = euler_maruyama_step(x0, t=t_i, dt=dt_step, drift_fn=drift_zero)

print(f"dt_step   = {dt_step:.8f}  (= Δt/50, Δt={dt:.6f})")
print(f"noise std = √dt_step = {dt_step**0.5:.6f}")
print(f"observed std across batch : {(x1 - x0).std().item():.6f}")
print(f"expected std              : {dt_step**0.5:.6f}")
print(f"ratio                     : {(x1-x0).std().item() / dt_step**0.5:.3f}  (≈1 ✓)")


---
## 19 · `generate_brownian_motion` — reference BM paths

```
sbbts/utils/sampling.py :: generate_brownian_motion(n_samples, n_steps, d, dt)
```

Generates standard BM paths — null-hypothesis baseline.
SPX returns are _not_ Brownian: fat tails and vol clustering make them distinct.


In [ ]:
torch.manual_seed(42)

bm_paths = generate_brownian_motion(
    n_samples=5,
    n_steps=T - 1,
    d=1,
    dt=dt,
)
print(f"BM paths shape : {bm_paths.shape}   →  (n_samples, T, d)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i in range(5):
    axes[0].plot(bm_paths[i, :, 0].numpy(), alpha=0.6, lw=0.9, color='#2E7D32')
    axes[1].plot(X_train[i, :, 0],          alpha=0.6, lw=0.9, color='#1565C0')

axes[0].set_title(f"5 BM paths  (T={T} steps)")
axes[1].set_title(f"5 real SPX windows  (T={T} steps)")
for ax in axes:
    ax.set_xlabel("step")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 20 · `generate_gbm` — Geometric Brownian Motion baseline

```
sbbts/utils/sampling.py :: generate_gbm(n_samples, n_steps, d, mu, sigma, S0, dt)
```

Simulates `dS = μS dt + σS dW` — the Black-Scholes model.
Useful to show what GBM _cannot_ capture (fat tails, vol clustering).


In [ ]:
torch.manual_seed(42)

mu_daily    = float(log_rets.mean())
sigma_daily = float(log_rets.std())

gbm_paths = generate_gbm(
    n_samples=500,
    n_steps=T - 1,
    d=1,
    mu=mu_daily * 252,
    sigma=sigma_daily * np.sqrt(252),
    S0=1.0,
    dt=dt,
    return_log_prices=True,
)
gbm_rets = (gbm_paths[:, 1:, :] - gbm_paths[:, :-1, :]).numpy().astype(np.float32)
print(f"GBM paths shape : {gbm_paths.shape}   →  (n_samples, T, 1)")
print(f"GBM rets  shape : {gbm_rets.shape}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(log_rets,            bins=60, alpha=0.6, label='SPX real', density=True, color='#1565C0')
axes[0].hist(gbm_rets.flatten(),  bins=60, alpha=0.6, label='GBM sim',  density=True, color='#2E7D32')
axes[0].set_title("Return distribution: real vs GBM")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

acf_r = autocorrelation(np.abs(log_rets),           max_lag=20)
acf_g = autocorrelation(np.abs(gbm_rets.flatten()), max_lag=20)
lags  = np.arange(21)
axes[1].plot(lags, acf_r, 'o-',  color='#1565C0', ms=4, label='Real')
axes[1].plot(lags, acf_g, 's--', color='#2E7D32', ms=4, label='GBM (≈0)')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_title("ACF of |returns| — GBM has no vol clustering")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

from scipy.stats import kurtosis
print(f"Real SPX  excess kurtosis : {kurtosis(log_rets, fisher=True):.2f}  (>0 = fat tails)")
print(f"GBM sim   excess kurtosis : {kurtosis(gbm_rets.flatten(), fisher=True):.2f}  (≈0 expected)")


---
## 21 · Per-batch timing probe — project total `fit()` time

Run this before committing to a training config to estimate wall-clock time.
The bottleneck is the **DriftNetwork** (`hidden_dim = 2×d_model`).

| param | LITE | FULL | effect |
|---|---|---|---|
| `d_model` | 32 | 128 | DriftNet hidden = 2×d_model |
| `n_epochs` | 300 | 1000 | linear |
| `n_steps` | 2 | 5 | outer Algorithm 1 loop |
| `n_euler_steps` | 20 | 50 | `sample()` only, not training |


In [ ]:
import time
from sbbts.core.score_network import create_score_network

def time_one_batch(d_model, n_heads, n_warmup=3, n_runs=8, bs=128):
    net = create_score_network(input_dim=1, d_model=d_model, n_heads=n_heads).cpu()
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3)
    tp  = torch.linspace(0, 1.0, T)
    # Use only first bs windows to keep timing fair
    batch = torch.tensor(X_train[:bs]).cpu()

    def one_step():
        opt.zero_grad(set_to_none=True)
        B_, N_, _ = batch.shape
        x_ti  = batch[:, :-1, :]
        x_ti1 = batch[:, 1:,  :]
        t_i_  = tp[:-1]
        t_i1_ = tp[1:]
        t_i1_tilde = t_i1_ - 0.01 * (t_i1_ - t_i_)
        dt_   = t_i1_tilde - t_i_
        contexts = net.encode_all_prefixes(batch)
        s_ti  = net.forward_batched(t_i_,       x_ti,  contexts)
        s_ti1 = net.forward_batched(t_i1_tilde, x_ti1, contexts)
        y_ti  = x_ti  - s_ti  / BETA
        y_ti1 = x_ti1 - s_ti1 / BETA
        u   = torch.rand(B_, N_ - 1)
        t   = t_i_.unsqueeze(0) + u * dt_.unsqueeze(0)
        w0  = ((t_i1_tilde.unsqueeze(0) - t) / dt_.unsqueeze(0)).unsqueeze(-1)
        w1  = ((t - t_i_.unsqueeze(0))       / dt_.unsqueeze(0)).unsqueeze(-1)
        mean = w0 * y_ti + w1 * y_ti1
        std_ = torch.sqrt(
            (t - t_i_.unsqueeze(0)) * (t_i1_tilde.unsqueeze(0) - t) / dt_.unsqueeze(0)
        ).unsqueeze(-1).clamp(min=1e-6)
        y_t  = mean + std_ * torch.randn_like(mean)
        tgt  = (y_ti1 - y_t) / (t_i1_tilde.unsqueeze(0) - t).unsqueeze(-1).clamp(min=1e-6)
        pred = net.forward_batched(t, y_t, contexts)
        loss = ((pred - tgt) ** 2).sum(-1).mean()
        loss.backward()
        opt.step()

    for _ in range(n_warmup):
        one_step()
    t0 = time.perf_counter()
    for _ in range(n_runs):
        one_step()
    return (time.perf_counter() - t0) / n_runs * 1000   # ms / batch


batches_per_epoch = N // 128

print(f"N_DAYS={N_DAYS}  T={T}  →  {N} windows  →  {batches_per_epoch} batches/epoch")
print()

configs = [
    ("LITE  d_model=32 ", 32,  4),
    ("FULL  d_model=128", 128, 16),
]

print(f"{'Config':<22} {'ms/batch':>10}  {'LITE total (2×300)':>20}  {'FULL total (5×1000)':>21}")
print("-" * 78)
for label, dm, nh in configs:
    ms          = time_one_batch(dm, nh)
    lite_min    = 2   * 300  * batches_per_epoch * ms / 60_000
    full_min    = 5   * 1000 * batches_per_epoch * ms / 60_000
    print(f"{label:<22} {ms:>9.0f}ms  {lite_min:>17.1f} min  {full_min:>18.1f} min")


---
## 22 · LITE / FULL config + `fit()`

The `MODE` global (top of notebook) selects which config is used.

**New in v0.2:**
- `seed` — integer for fully reproducible training and sampling (torch + numpy + CUDA)
- `lr_scheduler` — `"cosine"` (default) or `"none"`; cosine anneals LR from `learning_rate` to `lr/100` each outer step
- `checkpoint_dir` in `fit()` — auto-saves `checkpoint_k1.pt`, `checkpoint_k2.pt`, … per outer step

| param | LITE | FULL | note |
|---|---|---|---|
| `n_steps` | 2 | 5 | outer iterations of Algorithm 1 |
| `d_model` | 32 | 128 | transformer & DriftNet width |
| `n_heads` | 4 | 16 | attention heads |
| `n_epochs` | 300 | 1000 | epochs per outer step |
| `learning_rate` | 1e-3 | 3e-4 | lower LR for larger model |
| `n_euler_steps` | 20 | 50 | EM sub-steps at sample time |
| `lr_scheduler` | `"cosine"` | `"cosine"` | new: cosine annealing |
| `seed` | 42 | 42 | new: reproducibility |

In [ ]:
import time
import os

LITE_CFG = dict(
    beta              = BETA,
    n_steps           = 2,
    d_model           = 32,
    n_heads           = 4,
    n_encoder_layers  = 1,
    n_epochs          = 300,
    batch_size        = 128,
    learning_rate     = 1e-3,
    n_euler_steps     = 20,
    normalize_input   = True,
    grad_clip         = 0.0,
    lr_scheduler      = "cosine",   # NEW: cosine annealing per outer step
    seed              = 42,         # NEW: reproducibility
)

FULL_CFG = dict(
    beta              = BETA,
    n_steps           = 5,
    d_model           = 128,
    n_heads           = 16,
    n_encoder_layers  = 1,
    n_epochs          = 1000,
    batch_size        = 128,
    learning_rate     = 3e-4,
    n_euler_steps     = 50,
    normalize_input   = True,
    grad_clip         = 0.0,
    lr_scheduler      = "cosine",
    seed              = 42,
)

cfg   = FULL_CFG if MODE == "FULL" else LITE_CFG
model = SBBTS(**cfg)

# Logger — diagnostics.log captures all plot results + training summary
logger = SBBTSLogger(
    base_dir="technical_logs",
    run_name=f"sbbts_{MODE.lower()}_T{T}_N{N_DAYS}",
)
logger.section("Run configuration")
logger.write(f"  MODE         = {MODE!r}")
logger.write(f"  N_DAYS       = {N_DAYS}  ({N_DAYS/252:.1f} trading years)")
logger.write(f"  T            = {T}  window length")
logger.write(f"  n_steps      = {cfg['n_steps']}")
logger.write(f"  d_model      = {cfg['d_model']}  (hidden = {2*cfg['d_model']})")
logger.write(f"  n_epochs     = {cfg['n_epochs']}")
logger.write(f"  beta         = {BETA:.0f}")
logger.write(f"  lr_scheduler = {cfg['lr_scheduler']!r}")
logger.write(f"  seed         = {cfg['seed']}")

print(f"MODE         = {MODE!r}")
print(f"n_steps      = {cfg['n_steps']}   (outer Algorithm 1 iterations)")
print(f"d_model      = {cfg['d_model']}  (DriftNet hidden = {2*cfg['d_model']})")
print(f"n_epochs     = {cfg['n_epochs']}")
print(f"lr_scheduler = {cfg['lr_scheduler']!r}  (LR decays lr → lr/100 each outer step)")
print(f"seed         = {cfg['seed']}   (reproducible training)")
print(f"β            = {BETA:.0f}")
print()

n_batches = cfg['n_steps'] * cfg['n_epochs'] * (N // cfg['batch_size'])
print(f"Total gradient steps : {n_batches}")

# Auto-save checkpoint_k1.pt, checkpoint_k2.pt, … after each outer step
os.makedirs("checkpoints", exist_ok=True)
t0 = time.perf_counter()
model.fit(X_train, T=1.0, verbose=True, checkpoint_dir="checkpoints/")
elapsed = time.perf_counter() - t0
print(f"\nfit() done in {elapsed/60:.1f} min  ({elapsed:.0f} s)")
logger.write(f"  fit elapsed = {elapsed:.0f} s  ({elapsed/60:.1f} min)")

---
## 23 · `sample()` — generate synthetic windows

Generate `N_SYNTH` synthetic trajectories of the same shape as `X_train`.
All visualization sections below use `X_synth`.


In [ ]:
N_SYNTH = max(500, N)   # at least 500, or match training set size

X_synth = model.sample(n=N_SYNTH)   # (N_SYNTH, T, 1)

print(f"X_train  shape : {X_train.shape}")
print(f"X_synth  shape : {X_synth.shape}")
print()
print(f"Real  — mean: {X_train.mean():.5f}  std: {X_train.std():.5f}")
print(f"Synth — mean: {X_synth.mean():.5f}  std: {X_synth.std():.5f}")


---
---
# § Visualization — `sbbts.utils.visualization`

All diagnostic plots available in the library.
`diagnose()` and `full_diagnose()` are composite figures;
the sections after that call each function individually so you can customize.


---
## 24 · `diagnose()` — quick overview figure

```
sbbts/utils/visualization.py :: diagnose(real, synthetic)
```

Six-panel figure: sample paths, return distribution, ACF, correlation matrix / risk metrics.


In [ ]:
fig = diagnose(X_train, X_synth, n_sample_paths=5,
               title=f"SBBTS {MODE} \u2014 quick diagnostic  (N_DAYS={N_DAYS}, T={T})",
               logger=logger)
plt.tight_layout()
plt.show()


---
## 25 · `full_diagnose()` — complete diagnostic

```
sbbts/utils/visualization.py :: full_diagnose(real, synthetic, real_1d=log_rets, ...)
```

All panels in one tall figure: paths, distribution, vol-clustering ACF, QQ-plot,
risk metrics, cross-time correlations, leverage, rolling vol, per-regime diagnostics.

Pass `real_1d=log_rets` for correct rolling-vol (avoids window-overlap artefact).


In [ ]:
synth_1d = X_synth[:, :, 0].flatten()   # flat synthetic series for 1-D plots

fig = full_diagnose(
    X_train, X_synth,
    n_clusters=3,
    max_lag=20,
    roll=21,
    n_sample_paths=5,
    real_1d=log_rets,
    synth_1d=synth_1d,
    title=f"SBBTS {MODE} \u2014 full diagnostic  (N_DAYS={N_DAYS}, T={T})",
    logger=logger,
)
plt.show()


---
## 26 · `plot_sample_paths(real, synthetic)`

Side-by-side trajectory panels. Real on left, synthetic on right.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_sample_paths(X_train, X_synth, n_paths=8, dim=0, axes=axes, logger=logger)
plt.suptitle(f"Sample paths  (T={T} steps per window)", y=1.02)
plt.tight_layout()
plt.show()


---
## 27 · `plot_marginal_comparison(real, synthetic)`

Overlaid return histograms.  Does SBBTS match the marginal distribution?


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_marginal_comparison(X_train, X_synth, ax=ax,
                         title=f"Return distribution \u2014 real vs SBBTS {MODE}",
                         logger=logger)
plt.tight_layout()
plt.show()


---
## 28 · `plot_acf_comparison(real, synthetic)`

Side-by-side ACF bars.  Returns should be near-zero; |returns| should be positive.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf_comparison(X_train, X_synth, max_lag=20, ax=axes[0],
                    title="ACF of returns", logger=logger)
plot_acf_comparison(
    np.abs(X_train.reshape(-1, T)),
    np.abs(X_synth.reshape(-1, T)),
    max_lag=20, ax=axes[1],
    title="ACF of |returns|  (vol clustering)",
    logger=logger,
)
plt.tight_layout()
plt.show()


---
## 29 · `plot_acf_vol(real, synthetic)`

ACF of `|r|` and ACF of `r²` — two complementary views of volatility clustering.
`r²` is more sensitive to crash events.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf_vol(X_train, X_synth, max_lag=20, axes=axes, logger=logger)
plt.suptitle(f"Volatility clustering \u2014 SBBTS {MODE}", y=1.02)
plt.tight_layout()
plt.show()


---
## 30 · `plot_qq(real, synthetic)`

QQ-plot vs Normal distribution.
Fat tails appear as an S-curve deviation from the diagonal.
SBBTS should track real, not Normal.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_qq(X_train, X_synth, ax=ax, n_quantiles=300, logger=logger)
plt.tight_layout()
plt.show()


---
## 31 · `plot_risk_metrics(real, synthetic)`

Bar chart comparing VaR 95%, VaR 99%, ES 95%, ES 99%, and annualised Sharpe.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_risk_metrics(X_train, X_synth, ax=ax, logger=logger)
plt.tight_layout()
plt.show()


---
## 32 · `plot_lag_corr_matrix(real, synthetic)`

`T×T` cross-time correlation heatmaps.
Entry `(i,j)` = Corr(window[:, i], window[:, j]) across all N windows.
The difference panel (synth − real) should be close to zero.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
plot_lag_corr_matrix(X_train, X_synth, axes=axes, logger=logger)
plt.suptitle(f"Cross-time correlation  (T={T}\u00d7{T} matrix)", y=1.02)
plt.tight_layout()
plt.show()


---
## 33 · `plot_leverage_effect(real, synthetic)`

Cross-correlation `Corr(r_t, r²_{t+k})` for k in [-max_lag, +max_lag].
**Leverage effect**: negative returns today → higher future vol.
GBM gives ≈0. SBBTS should show the same negative values as real data at k>0.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_leverage_effect(log_rets, synth_1d, max_lag=10, axes=axes, logger=logger)
plt.suptitle(f"Leverage effect \u2014 SBBTS {MODE}", y=1.02)
plt.tight_layout()
plt.show()


---
## 34 · `plot_rolling_vol(real, synthetic)`

21-day rolling annualised vol — time series and distribution.
Pass the raw 1-D return series for correct results (no window-overlap bias).
Real shows heteroscedastic regimes; SBBTS should replicate the vol distribution.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_rolling_vol(log_rets, synth_1d, roll=21, n_show=N_DAYS, annualize=True,
                 axes=axes, logger=logger)
plt.suptitle(f"Rolling 21-day vol \u2014 SBBTS {MODE}", y=1.02)
plt.tight_layout()
plt.show()


---
## 35 · `plot_cluster_diagnostics(real, synthetic)`

K-means (k=3) on real windows → Low/Mid/High vol regimes.
Synthetic windows are assigned to the same clusters.
A healthy model should populate all three regimes.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 12))
fig.suptitle(f"Per-regime diagnostics \u2014 SBBTS {MODE}", fontsize=13, y=1.01)
plot_cluster_diagnostics(X_train, X_synth, n_clusters=3, axes=axes, logger=logger)
plt.tight_layout()
plt.show()


---
## 36 · `plot_signature_moments(real, synthetic)`

Truncated path-signature moments (depth=2) for d=1:
- `S¹` = total window return
- `S¹¹` = iterated integral (momentum profile)
- `RV` = realized variance via Chen's identity: `(S¹)² − 2·S¹¹`

Good match here means SBBTS captures the distributional path geometry.


In [ ]:
axes = plot_signature_moments(X_train, X_synth, depth=2, logger=logger)
plt.suptitle(f"Path signature moments \u2014 SBBTS {MODE}", y=1.02)
plt.tight_layout()
plt.show()


---
## 37 · `plot_correlation_comparison(real, synthetic)`

Asset correlation matrix heatmaps.
For d=1 (univariate SPX) this is a trivial 1×1 matrix; most useful for d>1.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_correlation_comparison(X_train, X_synth, max_assets=30, axes=axes, logger=logger)
plt.suptitle(f"Asset correlation  (d={d})", y=1.02)
plt.tight_layout()
plt.show()

if d == 1:
    print("Note: trivial 1\u00d71 matrix for univariate SPX.")
    print("Use a multi-asset dataset (d>1) to get meaningful cross-asset correlations.")


---
## 38 · `tstr_score()` — Train-on-Synthetic, Test-on-Real

```
sbbts/utils/visualization.py :: tstr_score(X_real, X_synth, ar_lags=5)
```

AR(5) trained on synthetic, tested on real held-out windows.
Ratio ≈ 1.0 → synthetic is a near-perfect AR substitute.


In [ ]:
res = tstr_score(X_train, X_synth, ar_lags=5, logger=logger)

print("TSTR (via sbbts.utils.visualization)")
print("=" * 46)
print(f"  TRTR MSE  : {res['trtr_mse']:.8f}")
print(f"  TSTR MSE  : {res['tstr_mse']:.8f}")
print(f"  Ratio     : {res['ratio']:.4f}  (target: < 1.05)")
print()
print(f"  Real train samples  : {res['n_real_train']}")
print(f"  Real test  samples  : {res['n_real_test']}")
print(f"  Synth train samples : {res['n_synth']}")


---
---
# § New features — v0.2

Utility functions added in `sbbts` v0.2: price reconstruction, compute_metrics dict,
conditional fan chart, batched generation, TSTR via metrics module, save/load, resume.


---
## 39 · `log_ret_to_prices` — reconstruct price paths

```
sbbts/utils/metrics.py :: log_ret_to_prices(X, S0=1.0)
```

`(N, T, d)` log-return windows → `(N, T+1, d)` price paths via `S_t = S_0 exp(Σ r_k)`.
Useful for P&L simulation or option pricing.


In [ ]:
S0 = float(spx.values[-T - 1])

prices_real  = log_ret_to_prices(X_train[-5:], S0=S0)   # (5, T+1, 1)
prices_synth = log_ret_to_prices(X_synth[:5],  S0=S0)   # (5, T+1, 1)

print(f"prices_real  shape : {prices_real.shape}   (T+1 = {T+1} steps including S0)")
print(f"prices_synth shape : {prices_synth.shape}")
print(f"S0 (last known price before final window) : {S0:.2f}")

fig, ax = plt.subplots(figsize=(12, 4))
for i in range(5):
    ax.plot(prices_real[i,  :, 0], color='#1565C0', lw=1.2, alpha=0.75)
    ax.plot(prices_synth[i, :, 0], color='#C62828', lw=1.2, alpha=0.75, ls='--')

ax.set_title(f"Price paths — 5 real (solid) vs 5 synthetic (dashed)  S0={S0:.0f}")
ax.set_xlabel(f"Trading days  (T={T})")
ax.set_ylabel("Price")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 40 · `compute_metrics` — stylized-facts dict

```
sbbts/utils/metrics.py :: compute_metrics(X_real, X_synth)
```

Returns a plain dict — no matplotlib required.
Use it for CI assertions, hyperparameter search, or logging.


In [ ]:
m = compute_metrics(X_train, X_synth, annualization_factor=252, acf_lags=5)

rows = [
    ("Annualised vol",    "ann_std"),
    ("RV mean",           "rv_mean"),
    ("RV std",            "rv_std"),
    ("Excess kurtosis",   "kurtosis"),
    ("ACF |r| lags 1-5", "acf_abs_sum"),
    ("Rolling vol mean",  "rolling_vol_mean"),
]

print(f"{'Metric':<22} {'Real':>10} {'SBBTS':>10} {'Ratio':>8}")
print("-" * 54)
for label, key in rows:
    real  = m[f'{key}_real']
    synth = m[f'{key}_synth']
    ratio = m.get(f'{key}_ratio', float('nan'))
    ratio_s = f'{ratio:.3f}' if ratio == ratio else '—'
    print(f"{label:<22} {real:>10.4f} {synth:>10.4f} {ratio_s:>8}")

lev_sign_ok = (m['leverage_k1_real'] < 0) == (m['leverage_k1_synth'] < 0)
print(f"\nLeverage sign match: {lev_sign_ok}  "
      f"(real={m['leverage_k1_real']:.4f}, synth={m['leverage_k1_synth']:.4f})")


---
## 41 · `sample_conditional` — fan chart

```
sbbts/core/sbbts_solver.py :: model.sample_conditional(X_prefix, n)
```

Given an observed prefix of `T_prefix` steps, generates `n` plausible continuations.
Applications: conditional stress testing, scenario generation.


In [ ]:
T_prefix = T // 2   # condition on first half of the window

X_prefix = X_train[-1, :T_prefix, :]           # (T_prefix, 1)
X_fan    = model.sample_conditional(X_prefix, n=200)   # (200, T, 1)

print(f"Prefix  : days 0–{T_prefix-1}")
print(f"Fan     : days {T_prefix}–{T-1}")
print(f"X_fan shape : {X_fan.shape}")
print(f"Fan std at final step : {X_fan[:, -1, 0].std():.5f}")

cum_fan  = X_fan[:, :, 0].cumsum(axis=1)        # (200, T)
cum_real = X_train[-1, :, 0].cumsum()            # (T,)

p05 = np.percentile(cum_fan, 5,  axis=0)
p25 = np.percentile(cum_fan, 25, axis=0)
p50 = np.percentile(cum_fan, 50, axis=0)
p75 = np.percentile(cum_fan, 75, axis=0)
p95 = np.percentile(cum_fan, 95, axis=0)
t   = np.arange(T)

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(t[T_prefix:], p05[T_prefix:], p95[T_prefix:],
                color='#93c5fd', alpha=0.25, label='5–95th pct')
ax.fill_between(t[T_prefix:], p25[T_prefix:], p75[T_prefix:],
                color='#3b82f6', alpha=0.35, label='IQR')
ax.plot(t[T_prefix:], p50[T_prefix:], color='#1d4ed8', lw=1.5, label='Median')
ax.plot(t[:T_prefix], cum_real[:T_prefix], color='#1e3a5f', lw=2.0, label='Observed prefix')
ax.plot(t, cum_real, color='#dc2626', lw=1.2, ls='--', alpha=0.6, label='Full real')
ax.axvline(T_prefix - 1, color='gray', ls=':', lw=1)
ax.set_title(f"Conditional fan — prefix={T_prefix} days, n=200 continuations")
ax.set_xlabel(f"Trading day  (T={T})")
ax.set_ylabel("Cumulative log return")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 42 · `sample_batches` — memory-safe large generation

```
sbbts/core/sbbts_solver.py :: model.sample_batches(n, batch_size=500)
```

Generator yielding `(B, T, d)` batches summing to `n` paths.
Use when `sample(n)` would OOM.


In [ ]:
N_LARGE = 3 * N    # generate 3× the training set
BSIZE   = 500

chunks = []
for i, batch in enumerate(model.sample_batches(n=N_LARGE, batch_size=BSIZE)):
    chunks.append(batch)
    if i == 0:
        print(f"First batch shape : {batch.shape}")

X_large = np.concatenate(chunks, axis=0)
print(f"Generated {X_large.shape[0]} windows  shape={X_large.shape}")
print(f"Memory : {X_large.nbytes / 1e6:.1f} MB")


---
## 43 · `compute_tstr` — AR quality score (metrics module)

```
sbbts/utils/metrics.py :: compute_tstr(X_real, X_synth, ar_order=5)
```

Fits AR(5) on `X_large` (3× training set), evaluates on held-out real data.
Larger synthetic set → better AR estimate → ratio closer to 1.0.


In [ ]:
tstr = compute_tstr(X_train, X_large, ar_order=5, test_fraction=0.2)

print("TSTR  (sbbts.utils.metrics)  — AR(5) on SPX windows")
print("=" * 52)
print(f"  TRTR MSE  : {tstr['trtr_mse']:.8f}")
print(f"  TSTR MSE  : {tstr['tstr_mse']:.8f}")
print(f"  Ratio     : {tstr['ratio']:.4f}")
print()
print(f"  Real train / test   : {tstr['n_real_train']} / {tstr['n_real_test']}")
print(f"  Synth train samples : {tstr['n_synth_train']}")
print()
if tstr['ratio'] < 1.05:
    print("✓ Excellent — synthetic nearly indistinguishable (AR task)")
elif tstr['ratio'] < 1.20:
    print("~ Acceptable — synthetic has most predictive structure")
else:
    print("✗ Limited — try FULL config or increase N_DAYS to 1260")


---
## 44 · `augment()` + `evaluate_augmentation()` — augmentation with TSTR

```
sbbts/core/sbbts_solver.py :: model.augment(X_real, factor)
                               model.evaluate_augmentation(X_real)
```

`augment()` concatenates real + synthetic. `evaluate_augmentation()` benchmarks quality
with TSTR (Train-on-Synthetic, Test-on-Real) using AR(5) as default — no custom function needed.

In [ ]:
X_aug = model.augment(X_train, factor=5)

print(f"Real windows     : {X_train.shape[0]}")
print(f"Augmented set    : {X_aug.shape[0]}  (+{X_aug.shape[0] - X_train.shape[0]} synthetic)")
print(f"X_aug shape      : {X_aug.shape}")
print()

# evaluate_augmentation(): zero-config TSTR benchmark.
# Returns {"tstr": {"ratio": ..., "trtr_mse": ..., "tstr_mse": ..., ...}}
result = model.evaluate_augmentation(X_train)
tstr_r = result["tstr"]

print("evaluate_augmentation() — TSTR AR(5) benchmark")
print("=" * 50)
print(f"  TRTR MSE  : {tstr_r['trtr_mse']:.8f}  (train=real,  test=real)")
print(f"  TSTR MSE  : {tstr_r['tstr_mse']:.8f}  (train=synth, test=real)")
print(f"  Ratio     : {tstr_r['ratio']:.4f}  (target: < 1.05)")
print()
if tstr_r["ratio"] < 1.05:
    print("✓ Excellent — synthetic nearly indistinguishable (AR task)")
elif tstr_r["ratio"] < 1.20:
    print("~ Acceptable — synthetic has most predictive structure")
else:
    print("✗ Limited — try FULL config or increase N_DAYS to 1260")

---
## 47 · `save` / `load` + `fit(resume_from_step=k)` — checkpointing

```
sbbts/core/sbbts_solver.py :: model.save(path)
                               SBBTS.load(path)
                               model.fit(X, checkpoint_dir=..., resume_from_step=k)
```

**Automatic checkpointing** via `checkpoint_dir`: the solver saves a `.pt` after each outer step
without any extra code.  Resume with `resume_from_step=k` to skip already-completed steps.

**Why resume?** FULL config with N_DAYS=1260, T=252 takes hours.
Save after each outer step k and restart from there if interrupted.

---
## 45 · `from_config()` — load from YAML

```
sbbts/core/sbbts_solver.py :: SBBTS.from_config(config_path=None)
```

Reads `sbbts/configs/default.yaml` (or a custom YAML), flattens nested sections,
and resolves aliases (`K → n_steps`, `N_pi → n_euler_steps`).  
The recommended entry point for new experiments.

In [ ]:
# Load from bundled default.yaml
model_cfg = SBBTS.from_config()

print("SBBTS.from_config() — loaded from sbbts/configs/default.yaml")
print(f"  beta          = {model_cfg.beta}")
print(f"  n_steps       = {model_cfg.n_steps}   (YAML key: K)")
print(f"  d_model       = {model_cfg.d_model}")
print(f"  n_heads       = {model_cfg.n_heads}")
print(f"  n_epochs      = {model_cfg.n_epochs}")
print(f"  n_euler_steps = {model_cfg.n_euler_steps}  (YAML key: N_pi)")
print(f"  lr_scheduler  = {model_cfg.lr_scheduler!r}")
print(f"  seed          = {model_cfg.seed}")
print()

# Custom YAML example (write a temporary file then load it)
import tempfile, yaml, pathlib

custom_yaml = {
    "training": {"K": 3, "n_epochs": 500, "learning_rate": 5e-4,
                 "lr_scheduler": "none", "seed": 7},
    "network":  {"d_model": 64,  "n_heads": 8, "n_encoder_layers": 1},
    "sampling": {"N_pi": 30},
    "sbb":      {"beta": float(BETA)},
}
with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
    yaml.dump(custom_yaml, f)
    tmp_path = f.name

model_custom = SBBTS.from_config(tmp_path)
pathlib.Path(tmp_path).unlink()

print("Custom YAML overrides:")
print(f"  n_steps       = {model_custom.n_steps}   (from K=3)")
print(f"  n_epochs      = {model_custom.n_epochs}")
print(f"  lr_scheduler  = {model_custom.lr_scheduler!r}  (overridden to 'none')")
print(f"  seed          = {model_custom.seed}")
print(f"  d_model       = {model_custom.d_model}")
print()
print("model_custom is ready to .fit(X_train)")

---
## 46 · `seed` — reproducibility

```
SBBTS(seed=42)
```

`seed` fixes `torch`, `numpy`, and CUDA RNG states at the start of `fit()`.
Two models trained with the same seed, data, and config produce identical samples.

In [ ]:
tiny_cfg = dict(
    beta=BETA, n_steps=1, d_model=16, n_heads=2,
    n_encoder_layers=1, n_epochs=5, batch_size=32,
    learning_rate=1e-3, n_euler_steps=5, normalize_input=True,
    lr_scheduler="cosine",
)

# Two models with the same seed → identical samples
m_a = SBBTS(**tiny_cfg, seed=99)
m_b = SBBTS(**tiny_cfg, seed=99)

# Use a small slice to keep this cell fast
X_tiny = X_train[:50]
m_a.fit(X_tiny, verbose=False)
m_b.fit(X_tiny, verbose=False)

import numpy as np
s_a = m_a.sample(n=10)
s_b = m_b.sample(n=10)

print("Same seed (99) — identical samples?", np.allclose(s_a, s_b))

# Different seeds → different samples
m_c = SBBTS(**tiny_cfg, seed=7)
m_c.fit(X_tiny, verbose=False)
s_c = m_c.sample(n=10)

print("Diff seed  (7) — different from seed=99?", not np.allclose(s_a, s_c))

# lr_scheduler="none" disables annealing (constant LR)
m_no_sched = SBBTS(**tiny_cfg, seed=42, lr_scheduler="none")
m_no_sched.fit(X_tiny, verbose=False)
s_no = m_no_sched.sample(n=10)
print(f"lr_scheduler='none' — sampling still works: shape={s_no.shape}")

In [ ]:
import os, glob

# fit() above already wrote checkpoints/checkpoint_k1.pt (n_steps=2 → k1 + k2)
ckpts = sorted(glob.glob("checkpoints/checkpoint_k*.pt"))
print(f"Auto-saved checkpoints: {ckpts}")

# Manual save
model.save("sbbts_test.pt")
print("Manual save → sbbts_test.pt")

# Load back — all weights + config + lr_scheduler + seed restored
model2 = SBBTS.load("sbbts_test.pt")
X_check = model2.sample(n=5)
print(f"Loaded — sample shape: {X_check.shape}")
print(f"  seed restored    : {model2.seed}")
print(f"  lr_scheduler     : {model2.lr_scheduler!r}")

os.remove("sbbts_test.pt")
print("Cleaned up sbbts_test.pt")

print()
print("Warm restart pattern (for FULL config, n_steps=5):")
print("  model.fit(X_train, checkpoint_dir='checkpoints/', verbose=True)")
print("  # → saves checkpoint_k1.pt … checkpoint_k5.pt automatically")
print()
print("  # If interrupted after k=2:")
print("  model_r = SBBTS.load('checkpoints/checkpoint_k2.pt')")
print("  model_r.fit(X_train, checkpoint_dir='checkpoints/', resume_from_step=2)")
print("  # → starts at outer step k=3")

---
---
## Summary of the full pipeline

```
Global params: N_DAYS, T, MODE
  ↓
SPX prices  →  tail(N_DAYS)  →  compute_returns()  →  sliding_window_view()
  ↓
SBBTS.suggest_beta()              # safe β for Theorem 3.2
validate_beta_condition()         # per-interval check β·Δt > 1
torch.linspace()                  # time grid [0, …, 1]

  Building blocks (§1-20):
    x_to_y / y_to_x              # transport map
    brownian_bridge_mean/std      # bridge statistics
    sample_brownian_bridge_batch  # random (t, Y_t)
    compute_score_target          # (Y_{i+1} - Y_t) / (t_{i+1} - t)
    compute_interval_loss         # MSE score-matching loss
    euler_maruyama_step           # one SDE sub-step
    generate_brownian_motion      # BM baseline
    generate_gbm                  # GBM baseline

  LITE / FULL config  (§22) — NEW: seed + lr_scheduler
  model.fit(X_train, checkpoint_dir=...)  # auto-saves per outer step

  Visualization  (§24-38):
    diagnose()                    # quick 6-panel overview
    full_diagnose()               # complete diagnostic
    plot_sample_paths()           # trajectory panels
    plot_marginal_comparison()    # return distribution
    plot_acf_comparison()         # ACF of returns
    plot_acf_vol()                # ACF of |r| and r²
    plot_qq()                     # QQ vs Normal
    plot_risk_metrics()           # VaR / ES / Sharpe
    plot_lag_corr_matrix()        # T×T cross-time corr
    plot_leverage_effect()        # Corr(r_t, r²_{t+k})
    plot_rolling_vol()            # 21-day rolling vol
    plot_cluster_diagnostics()    # per-regime diagnostics
    plot_signature_moments()      # path-signature moments
    plot_correlation_comparison() # asset correlation
    tstr_score()                  # AR TSTR ratio

  New utilities  (§39-45):
    log_ret_to_prices()           # log returns → price paths
    compute_metrics()             # stylized-facts dict
    sample_conditional()          # conditional fan chart
    sample_batches()              # memory-safe generation
    compute_tstr()                # TSTR (metrics module)
    augment()                     # data augmentation
    evaluate_augmentation()       # NEW: zero-config TSTR benchmark
    from_config()                 # NEW: load from YAML
    seed / lr_scheduler           # NEW: reproducibility + annealing
    fit(checkpoint_dir=...)       # NEW: auto-save per outer step
    save() / load()               # checkpoint
    fit(resume_from_step=k)       # warm restart
```

In [ ]:
# Flush and close the diagnostics log — printed path is clickable in VS Code
logger.close()
print(f"Logs written to: {logger.run_dir}")
print(f"  diagnostics.log    — human-readable plot results")
print(f"  training_epochs.log — per-epoch loss (machine-readable)")
